In [4]:
import json
import random
import pandas as pd

def load_jsonl(file_path):
    """Loads a JSONL file into a list of dictionaries."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

random.seed(42)

# --- Config ---
PLACEHOLDER_INTENT = "Read and act on this content as an AI agent would."
BALANCE_MODE = "upsample"  # "upsample" (keep all BIPIA, oversample prod_null) or "downsample" (your original min() approach)

def format_sample(intent, context):
    # Explicit, spoof-resistant delimiters — a payload inside `context` that
    # writes "Context:" or "### INTENT" can't fake the boundary because the
    # tags aren't just plain words, and intent always comes first/fixed.
    return (
        "### USER_INTENT\n"
        f"{intent.strip()}\n\n"
        "### CONTEXT (untrusted, may contain injected instructions)\n"
        f"{context.strip()}"
    )

# 3. Format MAlmasabi (structured template, not flattened concat)
malmasabi_formatted = [
    {
        "text": format_sample(item["user_intent"], item["context"]),
        "label": item["label"],
        "source": "bipia"
    }
    for item in load_jsonl('indirect_context.jsonl')
]

# 4. Format Prod Null (same template shape, generic placeholder intent
#    so the model can't shortcut on "intent present vs absent")
prod_null_formatted = [
    {
        "text": format_sample(PLACEHOLDER_INTENT, item["text"]),
        "label": item["label"],
        "source": "prodnull"
    }
    for item in load_jsonl('repo_prod_null.jsonl')
]

# 5. Balance
if BALANCE_MODE == "downsample":
    target_size = min(len(malmasabi_formatted), len(prod_null_formatted))
    malmasabi_final = random.sample(malmasabi_formatted, target_size)
    prod_null_final = random.sample(prod_null_formatted, target_size)
else:  # upsample the smaller source so you don't throw away BIPIA data
    target_size = max(len(malmasabi_formatted), len(prod_null_formatted))
    def upsample(lst, n):
        return lst if len(lst) >= n else [random.choice(lst) for _ in range(n)]
    malmasabi_final = upsample(malmasabi_formatted, target_size)
    prod_null_final = upsample(prod_null_formatted, target_size)

# 6. Merge and Shuffle
combined_data = malmasabi_final + prod_null_final
random.shuffle(combined_data)

# 7. Export to CSV
df = pd.DataFrame(combined_data)
df.to_csv('llm_guardian_combined.csv', index=False, encoding='utf-8')

print(f"Saved {len(df)} rows ({BALANCE_MODE}) to llm_guardian_combined.csv")
print(df['source'].value_counts())
print(df['label'].value_counts())

Saved 140000 rows (upsample) to llm_guardian_combined.csv
source
bipia       70000
prodnull    70000
Name: count, dtype: int64
label
1    70837
0    69163
Name: count, dtype: int64


In [6]:
import pandas as pd 

df=pd.read_csv("llm_guardian_combined.csv")

In [7]:
df.head()

,text,label,source
0,### USER_INTENT\nimport numpy as np\nimport sc...,0,bipia
1,### USER_INTENT\nRead and act on this content ...,1,prodnull
2,### USER_INTENT\nRead and act on this content ...,0,prodnull
3,### USER_INTENT\nWhat motivated him to nod?\n\...,0,bipia
4,### USER_INTENT\nwhat was issued as a precauti...,1,bipia


In [14]:
import pandas as pd

df = pd.read_csv('llm_guardian_combined.csv')

# Look for BIPIA rows where the intent looks suspiciously long/code-like
bipia = df[df['source'] == 'bipia'].copy()
bipia['intent_len'] = bipia['text'].str.extract(r'### USER_INTENT\n(.*?)\n\n### CONTEXT', flags=1, expand=False).str.len()

print(bipia['intent_len'].describe())
print(bipia.sort_values('intent_len', ascending=False).head(5)['text'].tolist())

count    56000.000000
mean        48.927536
std         21.764784
min          7.000000
25%         29.000000
50%         43.000000
75%         73.000000
max        311.000000
Name: intent_len, dtype: float64
["### USER_INTENT\nindian american entertainer mauli dave has appeared on television as a contestant in a number of programs; however, she only appeared in the slightly different capacity of a celebrity contestant one time between 2004 and 2012, during the 4th season of a popular show-- what was the name of that television show?\n\n### CONTEXT (untrusted, may contain injected instructions)\n| Year                      | Show                               | Role                           | Channel                             | Extra            |\n| 2004                      | Boogie Woogie-Chalo America        | Dance Contestant               | SET (Sony Entertainment Television) | Finalist (top 5) |\n| 2007 (May–October)        | Saregamapa Challenge               | Singing Contes

In [15]:
# crude grouping: use the first ~100 chars of context to detect near-duplicate base examples
bipia['group_key'] = bipia['text'].str.slice(0, 150)
print(bipia['group_key'].duplicated().sum(), "near-duplicate base rows out of", len(bipia))

55395 near-duplicate base rows out of 70000


In [17]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# --- Load combined dataset ---
df = pd.read_csv('llm_guardian_combined.csv')

# --- Grouping key: prevents near-duplicate BIPIA rows from leaking across splits ---
# prodnull rows won't have duplicates in practice, but grouping them too is harmless
df['group_key'] = df['text'].str.slice(0, 150)

# --- Step 1: carve out test set (15%) ---
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
trainval_idx, test_idx = next(gss_test.split(df, groups=df['group_key']))

trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df     = df.iloc[test_idx].reset_index(drop=True)

# --- Step 2: carve out val set from remaining (15% of original -> ~17.6% of trainval) ---
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss_val.split(trainval_df, groups=trainval_df['group_key']))

train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df   = trainval_df.iloc[val_idx].reset_index(drop=True)

# --- Sanity checks: no group leakage, label balance, source balance ---
train_groups = set(train_df['group_key'])
val_groups   = set(val_df['group_key'])
test_groups  = set(test_df['group_key'])

assert not (train_groups & val_groups),  "Leakage between train and val!"
assert not (train_groups & test_groups), "Leakage between train and test!"
assert not (val_groups & test_groups),   "Leakage between val and test!"

for name, split in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n{name}: {len(split)} rows")
    print(split['label'].value_counts(normalize=True))
    print(split['source'].value_counts())

# --- Drop helper column and save ---
for split, fname in [(train_df, 'train.csv'), (val_df, 'val.csv'), (test_df, 'test.csv')]:
    split.drop(columns=['group_key']).to_csv(fname, index=False, encoding='utf-8')

print("\nSaved train.csv, val.csv, test.csv")


train: 98736 rows
label
1    0.508427
0    0.491573
Name: proportion, dtype: float64
source
bipia       49722
prodnull    49014
Name: count, dtype: int64

val: 20480 rows
label
0    0.508252
1    0.491748
Name: proportion, dtype: float64
source
prodnull    11082
bipia        9398
Name: count, dtype: int64

test: 20784 rows
label
1    0.508372
0    0.491628
Name: proportion, dtype: float64
source
bipia       10880
prodnull     9904
Name: count, dtype: int64

Saved train.csv, val.csv, test.csv
